In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from saliency_dataloader import build_obs_specs, build_dataset
from train.common.data import build_dataloader
from data_utils.data_loader_robomimic import GazePreprocessor
from omegaconf import OmegaConf
import torch

cfg = OmegaConf.load("configs/playground.yaml")

build_obs_specs(cfg.data)

dataset = build_dataset(cfg.data, gaze_ratio=1.0)

dataloader = build_dataloader(dataset, cfg.data, sampler=None, grad_accum_steps=1)

device = torch.device(cfg.training.device if torch.cuda.is_available() else 'cpu')
temporal_window = getattr(cfg.gaze, "temporal_window", None)

gaze_preprocessor = GazePreprocessor(
    img_height=cfg.data.img_height,
    img_width=cfg.data.img_width,
    gaze_sigma=cfg.gaze.mask_sigma,
    maxpoints=cfg.gaze.max_points,
    device=str(device),
    temporal_alpha=float(cfg.gaze.temporal_alpha),
    temporal_beta=float(cfg.gaze.temporal_beta),
    temporal_gamma=float(cfg.gaze.temporal_gamma),
    temporal_use_future=bool(cfg.gaze.temporal_use_future),
    temporal_window=temporal_window,
)


============= Initialized Observation Utils with Obs Spec =============

using obs modality: rgb with keys: ['image']
using obs modality: low_dim with keys: ['gaze_coords']
SequenceDataset: loading dataset into memory...
100%|██████████| 20/20 [00:00<00:00, 1633.87it/s]


In [10]:
batch = next(iter(dataloader))


In [11]:
print(batch["obs"]["image"].shape)
print(batch["obs"][cfg.data.gaze_key].shape)


torch.Size([64, 2, 180, 320, 3])
torch.Size([64, 2, 10])


In [12]:
obs_image_seq = batch['obs']['image'].to(device, non_blocking=True)
gaze_coords_cpu = batch['obs'][cfg.data.gaze_key]
gaze_coords_seq = gaze_coords_cpu.to(device, non_blocking=True)

obs_image, gaze_heatmaps, center_idx = gaze_preprocessor.prepare_for_bc(
    obs_image_seq=obs_image_seq,
    gaze_seq=gaze_coords_seq,
    frame_stack=cfg.data.frame_stack,
    grayscale=cfg.model.grayscale,
    aggregate_stack=bool(cfg.gaze.temporal_flag),
)

window = max(cfg.data.frame_stack, temporal_window or cfg.data.frame_stack)
gaze_window = gaze_preprocessor.extract_gaze_stack_around_center(
    gaze_coords_cpu,
    center_idx=center_idx,
    frame_stack=window,
)

print(f"obs_image shape: {obs_image.shape}")
print(f"gaze_heatmaps shape: {gaze_heatmaps.shape}")
print(f"center_idx: {center_idx}")
print(f"temporal window used: {window}")
print(f"gaze window shape: {tuple(gaze_window.shape)}")


obs_image shape: torch.Size([64, 2, 180, 320])
gaze_heatmaps shape: torch.Size([64, 2, 180, 320])
center_idx: 1
temporal window used: 4
gaze window shape: (64, 4, 10)


In [13]:
_, gaze_heatmaps_base, _ = gaze_preprocessor.prepare_for_bc(
    obs_image_seq=obs_image_seq,
    gaze_seq=gaze_coords_seq,
    frame_stack=cfg.data.frame_stack,
    grayscale=cfg.model.grayscale,
    aggregate_stack=False,
)
difference = (gaze_heatmaps - gaze_heatmaps_base).abs().mean().item()
print(f"mean |aggregated - per_frame|: {difference:.6f}")


mean |aggregated - per_frame|: 0.086217
